In [1]:
import pandas as pd

# Load the patient-level datasets created in Notebook 17

train_dataset = pd.read_csv(
    "../data/processed/sepsis_progression_train.csv"
)

validation_dataset = pd.read_csv(
    "../data/processed/sepsis_progression_validation.csv"
)

test_dataset = pd.read_csv(
    "../data/processed/sepsis_progression_test.csv"
)

print("Train shape:", train_dataset.shape)
print("Validation shape:", validation_dataset.shape)
print("Test shape:", test_dataset.shape)

Train shape: (300569, 58)
Validation shape: (64153, 58)
Test shape: (64226, 58)


In [2]:
### going through the columns
print("training columns:")
print(train_dataset.columns.tolist())
print("\ntarget distribution:")
print(train_dataset["progression_class"]
      .value_counts().sort_index())
print("\nMissing values:")
missing_table = pd.DataFrame({
    "missing_counts": train_dataset.isna().sum(),
    "missing_percent": train_dataset.isna().mean() * 100
})
print(missing_table.sort_values(
    "missing_percent",
    ascending=False
).head(20))

training columns:
['hour', 'heart_rate', 'resp_rate', 'temperature', 'sbp', 'dbp', 'mbp', 'spo2', 'gcs', 'creatinine', 'bun', 'urineoutput_last', 'urineoutput_sum', 'urineoutput_24hr', 'wbc', 'hemoglobin', 'hematocrit', 'platelet', 'bands', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'magnesium', 'aniongap', 'albumin', 'bilirubin_total', 'bilirubin_max', 'inr', 'pt', 'ptt', 'crp', 'lactate', 'pao2fio2ratio_novent', 'pao2fio2ratio_vent', 'glucose_lab', 'shock_index', 'map_calculated', 'bun_creatinine_ratio', 'spo2_deficit', 'heart_rate_prev', 'heart_rate_delta', 'resp_rate_prev', 'resp_rate_delta', 'temperature_prev', 'temperature_delta', 'sbp_prev', 'sbp_delta', 'mbp_prev', 'mbp_delta', 'spo2_prev', 'spo2_delta', 'gcs_prev', 'gcs_delta', 'heart_rate_roll3_mean', 'heart_rate_roll6_mean', 'progression_class']

target distribution:
progression_class
0     85039
1    146483
2     69047
Name: count, dtype: int64

Missing values:
                      missing_counts  missing

In [3]:
# Summary of missing values

missing_summary = pd.DataFrame({
    "missing_count": train_dataset.isna().sum(),
    "missing_percent": train_dataset.isna().mean() * 100
})

print("Features with <50% missing:")
print(
    missing_summary[
        missing_summary["missing_percent"] < 50
    ].sort_values("missing_percent")
)

print("\nFeatures with >=50% missing:")
print(
    missing_summary[
        missing_summary["missing_percent"] >= 50
    ].sort_values("missing_percent")
)

Features with <50% missing:
                       missing_count  missing_percent
hour                               0         0.000000
progression_class                  0         0.000000
heart_rate_roll6_mean             67         0.022291
heart_rate_roll3_mean            426         0.141731
heart_rate                      3193         1.062318
mbp                             4512         1.501153
sbp                             5093         1.694453
dbp                             5133         1.707761
map_calculated                  5160         1.716744
resp_rate                       5191         1.727058
heart_rate_prev                 5355         1.781621
shock_index                     5952         1.980244
resp_rate_prev                  7241         2.409097
spo2_deficit                    7350         2.445362
spo2                            7350         2.445362
heart_rate_delta                7587         2.524212
mbp_prev                        9277         3.086479


In [4]:
# Check data types

print("Data types:")
print(train_dataset.dtypes)

print("\nNon-numeric columns:")

non_numeric = train_dataset.select_dtypes(
    exclude=["number"]
).columns.tolist()

print(non_numeric)

Data types:
hour                       int64
heart_rate               float64
resp_rate                float64
temperature              float64
sbp                      float64
dbp                      float64
mbp                      float64
spo2                     float64
gcs                      float64
creatinine               float64
bun                      float64
urineoutput_last         float64
urineoutput_sum          float64
urineoutput_24hr         float64
wbc                      float64
hemoglobin               float64
hematocrit               float64
platelet                 float64
bands                    float64
sodium                   float64
potassium                float64
chloride                 float64
bicarbonate              float64
calcium                  float64
magnesium                float64
aniongap                 float64
albumin                  float64
bilirubin_total          float64
bilirubin_max            float64
inr                      float6

In [5]:
# Check target quality

print("Missing target values:")
print(train_dataset["progression_class"].isna().sum())

print("\nUnique target values:")
print(
    sorted(
        train_dataset["progression_class"].unique()
    )
)

print("\nTarget value counts:")
print(
    train_dataset["progression_class"]
    .value_counts()
    .sort_index()
)

Missing target values:
0

Unique target values:
[np.int64(0), np.int64(1), np.int64(2)]

Target value counts:
progression_class
0     85039
1    146483
2     69047
Name: count, dtype: int64


In [6]:
# Define features and target
target_column = "progression_class"

feature_columns = [
    column
    for column in train_dataset.columns
    if column != target_column
]

X_train = train_dataset[feature_columns].copy()
y_train = train_dataset[target_column].copy()

X_val = validation_dataset[feature_columns].copy()
y_val = validation_dataset[target_column].copy()

X_test = test_dataset[feature_columns].copy()
y_test = test_dataset[target_column].copy()

print("Number of features:", len(feature_columns))

print("\nTraining:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Number of features: 57

Training:
X_train: (300569, 57)
y_train: (300569,)

Validation:
X_val: (64153, 57)
y_val: (64153,)

Test:
X_test: (64226, 57)
y_test: (64226,)


In [ ]:
#Training:
#X_train: (300569, 57)
#y_train: (300569,)

### here the x denotes the rows and features and y is the targets

In [7]:
# Check LightGBM installation

import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.6.0


In [8]:
# Create baseline LightGBM model

model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=300, ### no of  decision tree
    learning_rate=0.05,   ### adjustment
    num_leaves=31,     ### controls the complexity of the tree
    random_state=42,
    n_jobs=-1   ### Use all available CPU cores
)

print(model)

LGBMClassifier(learning_rate=0.05, n_estimators=300, n_jobs=-1, num_class=3,
               objective='multiclass', random_state=42)


In [9]:
### trrain the baseline lgbm model
model.fit(
    X_train,
    y_train
)
print("Baseline LightGBM model trained successfully!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.009720 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9089
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 57
[LightGBM] [Info] Start training from score -1.262567
[LightGBM] [Info] Start training from score -0.718768
[LightGBM] [Info] Start training from score -1.470890
Baseline LightGBM model trained successfully!


In [10]:
### model prediction on the validation set
y_val_pred = model.predict(X_val)
print("Validation predictions generated successfully!")
print("prediction shape:", y_val_pred.shape)
print("first 20 predictions:", y_val_pred[:20])

Validation predictions generated successfully!
prediction shape: (64153,)
first 20 predictions: [1 0 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1]


In [11]:
### baseline model
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Calculate validation accuracy
accuracy = accuracy_score(y_val, y_val_pred)

print("Baseline Validation Accuracy:", accuracy)

# Detailed classification report
print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred,
        digits=4
    )
)

# Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

Baseline Validation Accuracy: 0.6202983492588032

Classification Report:
              precision    recall  f1-score   support

           0     0.7365    0.5979    0.6600     17927
           1     0.5892    0.9004    0.7123     31361
           2     0.5009    0.0565    0.1016     14865

    accuracy                         0.6203     64153
   macro avg     0.6089    0.5182    0.4913     64153
weighted avg     0.6099    0.6203    0.5562     64153


Confusion Matrix:
[[10718  7056   153]
 [ 2441 28236   684]
 [ 1393 12632   840]]


In [12]:
print("Actual validation distribution:")
print(y_val.value_counts().sort_index())

print("\nPredicted validation distribution:")
print(
    pd.Series(y_val_pred)
    .value_counts()
    .sort_index()
)

Actual validation distribution:
progression_class
0    17927
1    31361
2    14865
Name: count, dtype: int64

Predicted validation distribution:
0    14552
1    47924
2     1677
Name: count, dtype: int64


In [ ]:
### the model shows bias for the clss 1 which is stable class  and ignoring the class 2 which the deteriorating class which is a big problem .
#, the model was biased toward Class 1 (Stable) because it was the easiest way to maximize overall accuracy
### to avoid this bias ,  penalize mistakes on under-represented classes more heavily.

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight = "balanced",
    classes = classes,    ###Classes with fewer samples get a higher weight, and classes with many samples get a lower weight.
    y = y_train
)
print("Classes:", classes)
print("Balanced class weights:", class_weights)

for cls, weight in zip(classes, class_weights):
    print(f"Class {cls}: {weight:.4f}")  ### print  calculated class_weights array and prints them in a readable format, matching each class label (0, 1, or 2) with its corresponding weight.

Classes: [0 1 2]
Balanced class weights: [1.17816139 0.68396788 1.45103577]
Class 0: 1.1782
Class 1: 0.6840
Class 2: 1.4510


In [14]:
# class-weighted LightGBM model

weighted_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight={
        0: class_weights[0],
        1: class_weights[1],
        2: class_weights[2]
    },
    random_state=42,
    n_jobs=-1
)

print(weighted_model)

LGBMClassifier(class_weight={0: np.float64(1.178161392615937),
                             1: np.float64(0.6839678779562589),
                             2: np.float64(1.4510357679068846)},
               learning_rate=0.05, n_estimators=300, n_jobs=-1, num_class=3,
               objective='multiclass', random_state=42)


In [15]:
# Train class-weighted LightGBM model

weighted_model.fit(
    X_train,
    y_train
)

print("Class-weighted LightGBM model trained successfully!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010018 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9089
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 57
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
[LightGBM] [Info] Start training from score -1.098612
Class-weighted LightGBM model trained successfully!


In [17]:
### weighted model validation predictions
y_val_pred_weighted = weighted_model.predict(X_val)

print("Weighted model predictions generated successfully!")
print("Prediction shape:", y_val_pred_weighted.shape)

print("\nPredicted validation distribution:")
print(
    pd.Series(y_val_pred_weighted)
    .value_counts()
    .sort_index()
)

Weighted model predictions generated successfully!
Prediction shape: (64153,)

Predicted validation distribution:
0    22353
1    20713
2    21087
Name: count, dtype: int64


In [18]:
### weighted model validaton results
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

weighted_accuracy = accuracy_score(
    y_val,
    y_val_pred_weighted
)

print("Weighted Model Validation Accuracy:", weighted_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_weighted,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred_weighted
    )
)

Weighted Model Validation Accuracy: 0.5500132495752342

Classification Report:
              precision    recall  f1-score   support

           0     0.6101    0.7607    0.6771     17927
           1     0.6647    0.4390    0.5287     31361
           2     0.3737    0.5302    0.4384     14865

    accuracy                         0.5500     64153
   macro avg     0.5495    0.5766    0.5481     64153
weighted avg     0.5820    0.5500    0.5493     64153


Confusion Matrix:
[[13637  2502  1788]
 [ 6176 13767 11418]
 [ 2540  4444  7881]]


In [19]:
# Tuned LightGBM model
# No class weighting for this first tuning experiment
### tuning done in the learning rate andthen the complexity of the tree = num_leaves

tuned_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=-1,
    min_child_samples=30,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

print(tuned_model)

LGBMClassifier(colsample_bytree=0.8, learning_rate=0.03, min_child_samples=30,
               n_estimators=500, n_jobs=-1, num_class=3, num_leaves=63,
               objective='multiclass', random_state=42, subsample=0.8)


In [20]:
# Train tuned LightGBM model

tuned_model.fit(
    X_train,
    y_train
)

print("Tuned LightGBM model trained successfully!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010552 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9089
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 57
[LightGBM] [Info] Start training from score -1.262567
[LightGBM] [Info] Start training from score -0.718768
[LightGBM] [Info] Start training from score -1.470890
Tuned LightGBM model trained successfully!


In [21]:
# Generate validation predictions from tuned model

y_val_pred_tuned = tuned_model.predict(X_val)

print("Tuned model predictions generated successfully!")
print("Prediction shape:", y_val_pred_tuned.shape)

print("\nPredicted validation distribution:")
print(
    pd.Series(y_val_pred_tuned)
    .value_counts()
    .sort_index()
)

Tuned model predictions generated successfully!
Prediction shape: (64153,)

Predicted validation distribution:
0    14658
1    47457
2     2038
Name: count, dtype: int64


In [22]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

tuned_accuracy = accuracy_score(
    y_val,
    y_val_pred_tuned
)

print("Tuned Model Validation Accuracy:", tuned_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_tuned,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred_tuned
    )
)

Tuned Model Validation Accuracy: 0.6203762879366514

Classification Report:
              precision    recall  f1-score   support

           0     0.7357    0.6016    0.6619     17927
           1     0.5903    0.8932    0.7108     31361
           2     0.4921    0.0675    0.1187     14865

    accuracy                         0.6204     64153
   macro avg     0.6060    0.5207    0.4971     64153
weighted avg     0.6082    0.6204    0.5599     64153


Confusion Matrix:
[[10784  6973   170]
 [ 2484 28012   865]
 [ 1390 12472  1003]]


In [23]:
# LightGBM with increasing weight on the weight on class 2

moderate_weight_model = lgb.LGBMClassifier(
    objective="multiclass",
    num_class=3,
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight={
        0: 1.0,
        1: 1.0,
        2: 1.5
    },
    random_state=42,
    n_jobs=-1
)

print(moderate_weight_model)

LGBMClassifier(class_weight={0: 1.0, 1: 1.0, 2: 1.5}, learning_rate=0.05,
               n_estimators=300, n_jobs=-1, num_class=3, objective='multiclass',
               random_state=42)


In [24]:
moderate_weight_model.fit(
    X_train,
    y_train
)

print("Moderately weighted LightGBM model trained successfully!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011226 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 9089
[LightGBM] [Info] Number of data points in the train set: 300569, number of used features: 57
[LightGBM] [Info] Start training from score -1.371297
[LightGBM] [Info] Start training from score -0.827497
[LightGBM] [Info] Start training from score -1.174154
Moderately weighted LightGBM model trained successfully!


In [25]:
y_val_pred_moderate = moderate_weight_model.predict(X_val)

print("Moderately weighted model predictions generated successfully!")
print("Prediction shape:", y_val_pred_moderate.shape)

print("\nPredicted validation distribution:")
print(
    pd.Series(y_val_pred_moderate)
    .value_counts()
    .sort_index()
)

Moderately weighted model predictions generated successfully!
Prediction shape: (64153,)

Predicted validation distribution:
0    14433
1    39185
2    10535
Name: count, dtype: int64


In [26]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

moderate_accuracy = accuracy_score(
    y_val,
    y_val_pred_moderate
)

print("Moderate Weight Model Validation Accuracy:", moderate_accuracy)

print("\nClassification Report:")
print(
    classification_report(
        y_val,
        y_val_pred_moderate,
        digits=4
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_val,
        y_val_pred_moderate
    )
)

Moderate Weight Model Validation Accuracy: 0.6053029476407962

Classification Report:
              precision    recall  f1-score   support

           0     0.7393    0.5952    0.6595     17927
           1     0.6062    0.7574    0.6734     31361
           2     0.4185    0.2966    0.3472     14865

    accuracy                         0.6053     64153
   macro avg     0.5880    0.5497    0.5600     64153
weighted avg     0.5999    0.6053    0.5939     64153


Confusion Matrix:
[[10670  6328   929]
 [ 2411 23753  5197]
 [ 1352  9104  4409]]
